# ResNet-18 on Frequency-Domain Spectrogram Features

This version fixes the path/import issues and can auto-detect manifest filenames and FFT/spectrogram feature keys.
It can also generate transformed data if the manifests are missing.


In [1]:
import os
import time
from pathlib import Path

import matplotlib.pylab as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models import ResNet18_Weights

try:
    import wandb
except ImportError:
    wandb = None

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(device)


cpu


In [3]:
from pathlib import Path
import sys
import csv
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# --- Resolve repo paths and import transform helper ---
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "prototype-notebooks" else NOTEBOOK_DIR
TOOLS_DIR = PROJECT_ROOT / "tools"
OUTPUT_ROOT = PROJECT_ROOT / "output"

if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from data_transform import download_data

# --- Auto-detect data root / manifests ---
candidate_roots = [
    OUTPUT_ROOT / "spectrogram_data",
    OUTPUT_ROOT / "fft_data",
    OUTPUT_ROOT,
]

DATA_ROOT = None
for root in candidate_roots:
    if (root / "metadata").exists():
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    DATA_ROOT = OUTPUT_ROOT / "spectrogram_data"

METADATA_DIR = DATA_ROOT / "metadata"

def find_manifest(split_name: str) -> Path:
    candidates = [
        METADATA_DIR / f"{split_name}_fft_manifest.csv",
        METADATA_DIR / f"{split_name}_spectrogram_manifest.csv",
        METADATA_DIR / f"{split_name}_manifest.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

TRAIN_MANIFEST = find_manifest("train")
VAL_MANIFEST = find_manifest("validation")
TEST_MANIFEST = find_manifest("test")

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("TOOLS_DIR =", TOOLS_DIR)
print("DATA_ROOT =", DATA_ROOT)
print("METADATA_DIR exists =", METADATA_DIR.exists())
if METADATA_DIR.exists():
    print("Metadata files:")
    for p in sorted(METADATA_DIR.iterdir()):
        print(" -", p.name)

print("TRAIN_MANIFEST =", TRAIN_MANIFEST, TRAIN_MANIFEST.exists())
print("VAL_MANIFEST =", VAL_MANIFEST, VAL_MANIFEST.exists())
print("TEST_MANIFEST =", TEST_MANIFEST, TEST_MANIFEST.exists())

# --- Generate transformed data if manifests do not exist ---
DOWNLOAD_DATA = False   # set True to force regeneration
DOWNLOAD_SPLITS = ["train", "validation", "test"]
PAGE_NUM = 1
PAGE_SIZE = 100
EXPORT_ROOT = DATA_ROOT

if DOWNLOAD_DATA or not (TRAIN_MANIFEST.exists() and VAL_MANIFEST.exists() and TEST_MANIFEST.exists()):
    print("\nGenerating transformed data...")
    for split_name in DOWNLOAD_SPLITS:
        print(f"Generating transformed data for split: {split_name}")
        download_data(
            page_num=PAGE_NUM,
            page_size=PAGE_SIZE,
            split=split_name,
            export_root=EXPORT_ROOT,
        )

    TRAIN_MANIFEST = find_manifest("train")
    VAL_MANIFEST = find_manifest("validation")
    TEST_MANIFEST = find_manifest("test")

    print("\nAfter generation:")
    print("TRAIN_MANIFEST =", TRAIN_MANIFEST, TRAIN_MANIFEST.exists())
    print("VAL_MANIFEST =", VAL_MANIFEST, VAL_MANIFEST.exists())
    print("TEST_MANIFEST =", TEST_MANIFEST, TEST_MANIFEST.exists())

# --- Dataset helpers ---
class_names = ["real", "ai_generated"]
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

def load_manifest_rows(manifest_path: Path):
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")
    with manifest_path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def to_spectrogram_from_npz(sample) -> np.ndarray:
    keys = list(sample.keys())

    if "spectrogram_normalized" in keys:
        return sample["spectrogram_normalized"].astype(np.float32)

    if "spectrogram" in keys:
        spec = sample["spectrogram"].astype(np.float32)
    elif "log_power_spectrum_shifted" in keys:
        spec = sample["log_power_spectrum_shifted"].astype(np.float32)
    elif "power_spectrum_shifted" in keys:
        spec = np.log1p(sample["power_spectrum_shifted"].astype(np.float32))
    elif "magnitude_shifted" in keys:
        spec = np.log1p(sample["magnitude_shifted"].astype(np.float32))
    elif "fft_shifted_real" in keys and "fft_shifted_imag" in keys:
        real = sample["fft_shifted_real"].astype(np.float32)
        imag = sample["fft_shifted_imag"].astype(np.float32)
        spec = np.log1p(real**2 + imag**2)
    elif "fft_real" in keys and "fft_imag" in keys:
        real = sample["fft_real"].astype(np.float32)
        imag = sample["fft_imag"].astype(np.float32)
        spec = np.log1p(real**2 + imag**2)
    else:
        raise KeyError(f"Could not build spectrogram from keys: {keys}")

    mean = spec.mean()
    std = spec.std()
    if std < 1e-6:
        std = 1.0
    return ((spec - mean) / std).astype(np.float32)

class SpectrogramManifestDataset(Dataset):
    def __init__(self, manifest_path: Path, data_root: Path):
        self.manifest_path = Path(manifest_path)
        self.data_root = Path(data_root)
        self.rows = load_manifest_rows(self.manifest_path)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        rel_path = row.get("spectrogram_path") or row.get("fft_path") or row.get("image_path")
        if rel_path is None:
            raise KeyError(f"Manifest row missing path field. Row keys: {list(row.keys())}")

        npz_path = self.data_root / rel_path
        if not npz_path.exists():
            raise FileNotFoundError(f"Sample file not found: {npz_path}")

        with np.load(npz_path) as sample:
            feature = to_spectrogram_from_npz(sample)

        tensor = torch.from_numpy(feature)
        label = class_to_idx[row["label_a_name"]]
        return tensor, label

# --- Create datasets/loaders ---
batch_size = 16
num_workers = 8

train_dataset = SpectrogramManifestDataset(TRAIN_MANIFEST, DATA_ROOT)
val_dataset = SpectrogramManifestDataset(VAL_MANIFEST, DATA_ROOT)
test_dataset = SpectrogramManifestDataset(TEST_MANIFEST, DATA_ROOT)

trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
testloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("\nLoaded datasets:")
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))
print("First train row:", train_dataset.rows[0])


NOTEBOOK_DIR = C:\Users\toshi\OneDrive\Documentos\ENSF617-Project\prototype-notebooks
PROJECT_ROOT = C:\Users\toshi\OneDrive\Documentos\ENSF617-Project
TOOLS_DIR = C:\Users\toshi\OneDrive\Documentos\ENSF617-Project\tools
DATA_ROOT = C:\Users\toshi\OneDrive\Documentos\ENSF617-Project\output\spectrogram_data
METADATA_DIR exists = True
Metadata files:
 - train_spectrogram_manifest.csv
TRAIN_MANIFEST = C:\Users\toshi\OneDrive\Documentos\ENSF617-Project\output\spectrogram_data\metadata\train_spectrogram_manifest.csv True
VAL_MANIFEST = C:\Users\toshi\OneDrive\Documentos\ENSF617-Project\output\spectrogram_data\metadata\validation_fft_manifest.csv False
TEST_MANIFEST = C:\Users\toshi\OneDrive\Documentos\ENSF617-Project\output\spectrogram_data\metadata\test_fft_manifest.csv False

Generating transformed data...
Generating transformed data for split: train
Downloaded page 1/420 for train: 100 rows
Downloaded page 2/420 for train: 100 rows
Downloaded page 3/420 for train: 100 rows
Downloaded pag

OSError: [Errno 28] No space left on device

# ResNet-18 on RGB FFT Log-Power Spectra

This notebook follows the same overall structure as the other classifier notebooks, but it uses `tools/data_transform.py`.

Workflow:
1. Optionally **pull and transform** Defactify rows into FFT `.npz` files using `tools/data_transform.py`
2. Read the generated FFT manifests
3. Build a **3-channel spectrogram-like input** from each saved sample
4. Train a **ResNet-18** classifier on that frequency-domain representation
5. Evaluate on validation and test manifests, then inspect class accuracy and the confusion matrix

`data_transform.py` currently saves FFT-domain arrays such as:
- `fft_real`, `fft_imag`
- `magnitude`, `magnitude_shifted`
- `power_spectrum`, `power_spectrum_shifted`
- `log_magnitude`, `log_magnitude_shifted`
- `phase`, `phase_shifted`
- `original_shape`

This notebook converts those saved FFT features into a model input by using the **shifted log-power spectrum**. If a precomputed `spectrogram_normalized` key exists in the `.npz`, it will use that. Otherwise it computes one on the fly from `power_spectrum_shifted`.


In [ ]:
import csv
import os
import time
from pathlib import Path

import matplotlib.pylab as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models import ResNet18_Weights

try:
    import wandb
except ImportError:
    wandb = None

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(device)


In [ ]:
print(class_names)
print("Train set:", len(train_dataset))
print("Val set:", len(val_dataset))
print("Test set:", len(test_dataset))

# Peek at one manifest row to make sure paths and labels look right.
print(train_dataset.rows[0])


In [ ]:
train_iterator = iter(trainloader)
train_batch = next(train_iterator)

print(train_batch[0].size())
print(train_batch[1].size())


In [ ]:
# Visualize one spectrogram-like sample from the training set.
sample_spec = train_batch[0][2].cpu().numpy()
sample_label = class_names[train_batch[1][2].item()]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
channel_titles = ["Red channel", "Green channel", "Blue channel"]

for i, ax in enumerate(axes):
    ax.imshow(sample_spec[i], cmap="magma")
    ax.set_title(channel_titles[i])
    ax.axis("off")

plt.suptitle(f"{sample_label} shifted log-power spectrum")
plt.tight_layout()
plt.show()


In [ ]:
class AiGenModel(nn.Module):
    def __init__(self, num_classes=2, use_pretrained=False, freeze_backbone=False):
        super().__init__()

        self.use_pretrained = use_pretrained
        self.freeze_backbone = freeze_backbone

        weights = ResNet18_Weights.DEFAULT if use_pretrained else None
        self.feature_extractor = models.resnet18(weights=weights)

        if self.use_pretrained and self.freeze_backbone:
            for name, param in self.feature_extractor.named_parameters():
                if not name.startswith('fc.'):
                    param.requires_grad = False

        in_features = self.feature_extractor.fc.in_features
        self.feature_extractor.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.feature_extractor(x)


In [ ]:
num_classes = len(class_names)
use_pretrained = False
freeze_backbone = False

net = AiGenModel(
    num_classes=num_classes,
    use_pretrained=use_pretrained,
    freeze_backbone=freeze_backbone,
).to(device)
print(net)


In [ ]:
criterion = nn.CrossEntropyLoss()
learning_rate = 1e-3
weight_decay = 1e-4

trainable_params = [param for param in net.parameters() if param.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

input_representation = 'rgb_fft_shifted_log_power_spectrum'

if WANDB_API_KEY and wandb is not None:
    wandb.login(key=WANDB_API_KEY)
else:
    print('W&B login skipped. Set WANDB_API_KEY and install wandb if you want remote logging.')

wandb_run = None
if wandb is not None:
    wandb_config = {
        'train_manifest': str(TRAIN_MANIFEST),
        'val_manifest': str(VAL_MANIFEST),
        'test_manifest': str(TEST_MANIFEST),
        'data_root': str(DATA_ROOT),
        'model_name': 'resnet18',
        'use_pretrained': use_pretrained,
        'freeze_backbone': freeze_backbone,
        'input_representation': input_representation,
        'batch_size': batch_size,
        'num_workers': num_workers,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'num_classes': num_classes,
        'device': str(device),
    }

    wandb_run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        config=wandb_config,
        mode='online' if WANDB_API_KEY else 'disabled',
    )

    wandb.define_metric('epoch')
    wandb.define_metric('train_loss', step_metric='epoch')
    wandb.define_metric('train_acc', step_metric='epoch')
    wandb.define_metric('val_loss', step_metric='epoch')
    wandb.define_metric('val_acc', step_metric='epoch')


In [ ]:
# Tune nepochs depending on how quickly validation loss plateaus.
nepochs = 10
PATH = './genai_resnet18_spectrogram_best.pth'

if wandb_run is not None:
    wandb.config.update({'nepochs': nepochs}, allow_val_change=True)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(nepochs):
    epoch_start_time = time.perf_counter()
    net.train()
    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in trainloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_dataset)
    train_acc = train_correct / train_total

    net.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in valloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = net(inputs)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / len(val_dataset)
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    epoch_time_sec = time.perf_counter() - epoch_start_time
    current_lr = optimizer.param_groups[0]['lr']

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(net.state_dict(), PATH)

    log_payload = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'lr': current_lr,
        'epoch_time_sec': epoch_time_sec,
    }

    if wandb_run is not None:
        wandb.log(log_payload)

    print(
        f"Epoch {epoch + 1}/{nepochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.6f} time={epoch_time_sec:.2f}s"
    )


In [ ]:
# Load the best model before evaluating on the test set.
net = AiGenModel(
    num_classes=len(class_names),
    use_pretrained=use_pretrained,
    freeze_backbone=freeze_backbone,
).to(device)
net.load_state_dict(torch.load(PATH, map_location=device))
net.eval()


In [ ]:
correct = 0
total = 0
test_loss = 0.0

test_start_time = time.perf_counter()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_inference_time_sec = time.perf_counter() - test_start_time
test_loss = test_loss / len(test_dataset)
test_accuracy = correct / total

test_metrics = {
    'test_loss': test_loss,
    'test_accuracy': test_accuracy,
    'test_accuracy_percent': 100 * test_accuracy,
    'test_inference_time_sec': test_inference_time_sec,
}

if wandb_run is not None:
    wandb.log(test_metrics)
    for key, value in test_metrics.items():
        wandb.run.summary[key] = value

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {100 * test_accuracy:.2f}%")
print(f"Test inference time: {test_inference_time_sec:.2f}s")


In [ ]:
# Optional: per-class accuracy.
class_correct = {name: 0 for name in class_names}
class_total = {name: 0 for name in class_names}

with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels, predicted):
            class_name = class_names[label.item()]
            class_total[class_name] += 1
            if label.item() == pred.item():
                class_correct[class_name] += 1

for class_name in class_names:
    accuracy = class_correct[class_name] / max(class_total[class_name], 1)
    print(f'{class_name}: {100 * accuracy:.2f}% ({class_correct[class_name]}/{class_total[class_name]})')
    if wandb_run is not None:
        wandb.log({f'test_accuracy_{class_name}': accuracy})
        wandb.run.summary[f'test_accuracy_{class_name}'] = accuracy


In [ ]:
# Optional: visualize training curves.
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('Accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Optional: plot the test-set confusion matrix.
confusion_matrix = torch.zeros((len(class_names), len(class_names)), dtype=torch.int64)

with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels.view(-1), predicted.view(-1)):
            confusion_matrix[label.item(), pred.item()] += 1

cm = confusion_matrix.cpu().numpy()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
fig.colorbar(im, ax=ax)

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Test Confusion Matrix')

threshold = cm.max() / 2 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            cm[i, j],
            ha='center',
            va='center',
            color='white' if cm[i, j] > threshold else 'black',
        )

plt.tight_layout()
plt.show()

if wandb_run is not None:
    wandb.finish()
